# 03 — Train Cyclegan / 生成对抗网络

**Chapter 26 — File 3 of 5 / 第26章 — 第3个文件（共5个）**

---

## Summary / 总结

This script demonstrates **example of training a cyclegan on the horse2zebra dataset**.

本脚本演示 **example of training a cyclegan on the horse2zebra dataset**。

---
## Step 1 — example of training a cyclegan on the horse2zebra dataset

In [ ]:
from random import random
from numpy import load
from numpy import zeros
from numpy import ones
from numpy import asarray
from numpy.random import randint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Concatenate
from tensorflow_addons.layers import InstanceNormalization

from matplotlib import pyplot

---
## Step 2 — define the discriminator model

In [ ]:
def define_discriminator(image_shape):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — source image input

In [ ]:
in_image = Input(shape=image_shape)

---
## Step 5 — C64

In [ ]:
d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 6 — C128

In [ ]:
d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 7 — C256

In [ ]:
d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 8 — C512

In [ ]:
d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 9 — second last output layer

In [ ]:
d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 10 — patch output

In [ ]:
patch_out = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)

---
## Step 11 — define model

In [ ]:
model = Model(in_image, patch_out)

---
## Step 12 — compile model

In [ ]:
model.compile(loss='mse', optimizer=Adam(learning_rate=0.0002, beta_1=0.5), loss_weights=[0.5])
	return model

---
## Step 13 — generator a resnet block

In [ ]:
def resnet_block(n_filters, input_layer):

---
## Step 14 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 15 — first layer convolutional layer

In [ ]:
g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(input_layer)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 16 — second convolutional layer

In [ ]:
g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)

---
## Step 17 — concatenate merge channel-wise with input layer

In [ ]:
g = Concatenate()([g, input_layer])
	return g

---
## Step 18 — define the standalone generator model

In [ ]:
def define_generator(image_shape, n_resnet=9):

---
## Step 19 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 20 — image input

In [ ]:
in_image = Input(shape=image_shape)

---
## Step 21 — c7s1-64

In [ ]:
g = Conv2D(64, (7,7), padding='same', kernel_initializer=init)(in_image)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 22 — d128

In [ ]:
g = Conv2D(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 23 — d256

In [ ]:
g = Conv2D(256, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 24 — R256

In [ ]:
for _ in range(n_resnet):
		g = resnet_block(256, g)

---
## Step 25 — u128

In [ ]:
g = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 26 — u64

In [ ]:
g = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 27 — c7s1-3

In [ ]:
g = Conv2D(3, (7,7), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	out_image = Activation('tanh')(g)

---
## Step 28 — define model

In [ ]:
model = Model(in_image, out_image)
	return model

---
## Step 29 — define a composite model for updating generators by adversarial and cycle loss

In [ ]:
def define_composite_model(g_model_1, d_model, g_model_2, image_shape):
    """Discriminators are compiled and will be trained separately, but generators are not compiled.
    """

---
## Step 30 — ensure the model we're updating is trainable

In [ ]:
g_model_1.trainable = True

---
## Step 31 — mark discriminator as not trainable

In [ ]:
d_model.trainable = False

---
## Step 32 — mark other generator model as not trainable

In [ ]:
g_model_2.trainable = False

---
## Step 33 — discriminator element

In [ ]:
input_gen = Input(shape=image_shape)
	gen1_out = g_model_1(input_gen)
	output_d = d_model(gen1_out)

---
## Step 34 — identity element

In [ ]:
input_id = Input(shape=image_shape)
	output_id = g_model_1(input_id)

---
## Step 35 — forward cycle

In [ ]:
output_f = g_model_2(gen1_out)

---
## Step 36 — backward cycle

In [ ]:
gen2_out = g_model_2(input_id)
	output_b = g_model_1(gen2_out)

---
## Step 37 — define model graph

In [ ]:
model = Model([input_gen, input_id], [output_d, output_id, output_f, output_b])

---
## Step 38 — define optimization algorithm configuration

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)

---
## Step 39 — compile model with weighting of least squares loss and L1 loss

In [ ]:
model.compile(loss=['mse', 'mae', 'mae', 'mae'], loss_weights=[1, 5, 10, 10], optimizer=opt)
	return model

---
## Step 40 — load and prepare training images

In [ ]:
def load_real_samples(filename):

---
## Step 41 — load the dataset

In [ ]:
data = load(filename)

---
## Step 42 — unpack arrays

In [ ]:
X1, X2 = data['arr_0'], data['arr_1']

---
## Step 43 — scale from [0,255] to [-1,1]

In [ ]:
X1 = (X1 - 127.5) / 127.5
	X2 = (X2 - 127.5) / 127.5
	return [X1, X2]

---
## Step 44 — select a batch of random samples, returns images and target

In [ ]:
def generate_real_samples(dataset, n_samples, patch_shape):

---
## Step 45 — choose random instances

In [ ]:
ix = randint(0, dataset.shape[0], n_samples)

---
## Step 46 — retrieve selected images

In [ ]:
X = dataset[ix]

---
## Step 47 — generate 'real' class labels (1)

In [ ]:
y = ones((n_samples, patch_shape, patch_shape, 1))
	return X, y

---
## Step 48 — generate a batch of images, returns images and targets

In [ ]:
def generate_fake_samples(g_model, dataset, patch_shape):

---
## Step 49 — generate fake instance

In [ ]:
X = g_model.predict(dataset)

---
## Step 50 — create 'fake' class labels (0)

In [ ]:
y = zeros((len(X), patch_shape, patch_shape, 1))
	return X, y

---
## Step 51 — save the generator models to file

In [ ]:
def save_models(step, g_model_AtoB, g_model_BtoA):

---
## Step 52 — save the first generator model

In [ ]:
filename1 = 'g_model_AtoB_%06d.h5' % (step+1)
	g_model_AtoB.save(filename1)

---
## Step 53 — save the second generator model

In [ ]:
filename2 = 'g_model_BtoA_%06d.h5' % (step+1)
	g_model_BtoA.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

---
## Step 54 — generate samples and save as a plot and save the model

In [ ]:
def summarize_performance(step, g_model, trainX, name, n_samples=5):

---
## Step 55 — select a sample of input images

In [ ]:
X_in, _ = generate_real_samples(trainX, n_samples, 0)

---
## Step 56 — generate translated images

In [ ]:
X_out, _ = generate_fake_samples(g_model, X_in, 0)

---
## Step 57 — scale all pixels from [-1,1] to [0,1]

In [ ]:
X_in = (X_in + 1) / 2.0
	X_out = (X_out + 1) / 2.0

---
## Step 58 — plot real images

In [ ]:
for i in range(n_samples):
		pyplot.subplot(2, n_samples, 1 + i)
		pyplot.axis('off')
		pyplot.imshow(X_in[i])

---
## Step 59 — plot translated image

In [ ]:
for i in range(n_samples):
		pyplot.subplot(2, n_samples, 1 + n_samples + i)
		pyplot.axis('off')
		pyplot.imshow(X_out[i])

---
## Step 60 — save plot to file

In [ ]:
filename1 = '%s_generated_plot_%06d.png' % (name, (step+1))
	pyplot.savefig(filename1)
	pyplot.close()

---
## Step 61 — update image pool for fake images

In [ ]:
def update_image_pool(pool, images, max_size=50):
	selected = list()
	for image in images:
		if len(pool) < max_size:

---
## Step 62 — stock the pool

In [ ]:
pool.append(image)
			selected.append(image)
		elif random() < 0.5:

---
## Step 63 — use image, but don't add it to the pool

In [ ]:
selected.append(image)
		else:

---
## Step 64 — replace an existing image and use replaced image

In [ ]:
ix = randint(0, len(pool))
			selected.append(pool[ix])
			pool[ix] = image
	return asarray(selected)

---
## Step 65 — train cyclegan models

In [ ]:
def train(d_model_A, d_model_B, g_model_AtoB, g_model_BtoA, c_model_AtoB, c_model_BtoA, dataset):

---
## Step 66 — define properties of the training run

In [ ]:
n_epochs, n_batch, = 100, 1

---
## Step 67 — determine the output square shape of the discriminator

In [ ]:
n_patch = d_model_A.output_shape[1]

---
## Step 68 — unpack dataset

In [ ]:
trainA, trainB = dataset

---
## Step 69 — prepare image pool for fakes

In [ ]:
poolA, poolB = list(), list()

---
## Step 70 — calculate the number of batches per training epoch

In [ ]:
bat_per_epo = int(len(trainA) / n_batch)

---
## Step 71 — calculate the number of training iterations

In [ ]:
n_steps = bat_per_epo * n_epochs

---
## Step 72 — manually enumerate epochs

In [ ]:
for i in range(n_steps):

---
## Step 73 — select a batch of real samples

In [ ]:
X_realA, y_realA = generate_real_samples(trainA, n_batch, n_patch)
		X_realB, y_realB = generate_real_samples(trainB, n_batch, n_patch)

---
## Step 74 — generate a batch of fake samples

In [ ]:
X_fakeA, y_fakeA = generate_fake_samples(g_model_BtoA, X_realB, n_patch)
		X_fakeB, y_fakeB = generate_fake_samples(g_model_AtoB, X_realA, n_patch)

---
## Step 75 — update fakes from pool

In [ ]:
X_fakeA = update_image_pool(poolA, X_fakeA)
		X_fakeB = update_image_pool(poolB, X_fakeB)

---
## Step 76 — update generator B->A via adversarial and cycle loss

In [ ]:
g_loss2, _, _, _, _  = c_model_BtoA.train_on_batch([X_realB, X_realA], [y_realA, X_realA, X_realB, X_realA])

---
## Step 77 — update discriminator for A -> [real/fake]

In [ ]:
dA_loss1 = d_model_A.train_on_batch(X_realA, y_realA)
		dA_loss2 = d_model_A.train_on_batch(X_fakeA, y_fakeA)

---
## Step 78 — update generator A->B via adversarial and cycle loss

In [ ]:
g_loss1, _, _, _, _ = c_model_AtoB.train_on_batch([X_realA, X_realB], [y_realB, X_realB, X_realA, X_realB])

---
## Step 79 — update discriminator for B -> [real/fake]

In [ ]:
dB_loss1 = d_model_B.train_on_batch(X_realB, y_realB)
		dB_loss2 = d_model_B.train_on_batch(X_fakeB, y_fakeB)

---
## Step 80 — summarize performance

In [ ]:
print('>%d, dA[%.3f,%.3f] dB[%.3f,%.3f] g[%.3f,%.3f]' % (i+1, dA_loss1,dA_loss2, dB_loss1,dB_loss2, g_loss1,g_loss2))

---
## Step 81 — evaluate the model performance every so often

In [ ]:
if (i+1) % (bat_per_epo * 1) == 0:

---
## Step 82 — plot A->B translation

In [ ]:
summarize_performance(i, g_model_AtoB, trainA, 'AtoB')

---
## Step 83 — plot B->A translation

In [ ]:
summarize_performance(i, g_model_BtoA, trainB, 'BtoA')
		if (i+1) % (bat_per_epo * 5) == 0:

---
## Step 84 — save the models

In [ ]:
save_models(i, g_model_AtoB, g_model_BtoA)

---
## Step 85 — load image data

In [ ]:
dataset = load_real_samples('horse2zebra_256.npz')
print('Loaded', dataset[0].shape, dataset[1].shape)

---
## Step 86 — define input shape based on the loaded dataset

In [ ]:
image_shape = dataset[0].shape[1:]

---
## Step 87 — generator: A -> B

In [ ]:
g_model_AtoB = define_generator(image_shape)

---
## Step 88 — generator: B -> A

In [ ]:
g_model_BtoA = define_generator(image_shape)

---
## Step 89 — discriminator: A -> [real/fake]

In [ ]:
d_model_A = define_discriminator(image_shape)

---
## Step 90 — discriminator: B -> [real/fake]

In [ ]:
d_model_B = define_discriminator(image_shape)

---
## Step 91 — composite: A -> B -> [real/fake, A]

In [ ]:
c_model_AtoB = define_composite_model(g_model_AtoB, d_model_B, g_model_BtoA, image_shape)

---
## Step 92 — composite: B -> A -> [real/fake, B]

In [ ]:
c_model_BtoA = define_composite_model(g_model_BtoA, d_model_A, g_model_AtoB, image_shape)

---
## Step 93 — train models

In [ ]:
train(d_model_A, d_model_B, g_model_AtoB, g_model_BtoA, c_model_AtoB, c_model_BtoA, dataset)

---
## Learning Notes / 学习笔记

- **概念**: example of training a cyclegan on the horse2zebra dataset 是机器学习中的常用技术。  
  *example of training a cyclegan on the horse2zebra dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Cyclegan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of training a cyclegan on the horse2zebra dataset
from random import random
from numpy import load
from numpy import zeros
from numpy import ones
from numpy import asarray
from numpy.random import randint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Concatenate
from tensorflow_addons.layers import InstanceNormalization

from matplotlib import pyplot

# define the discriminator model
def define_discriminator(image_shape):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# source image input
	in_image = Input(shape=image_shape)
	# C64
	d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	d = LeakyReLU(alpha=0.2)(d)
	# C128
	d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C256
	d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C512
	d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# second last output layer
	d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# patch output
	patch_out = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)
	# define model
	model = Model(in_image, patch_out)
	# compile model
	model.compile(loss='mse', optimizer=Adam(learning_rate=0.0002, beta_1=0.5), loss_weights=[0.5])
	return model

# generator a resnet block
def resnet_block(n_filters, input_layer):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# first layer convolutional layer
	g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(input_layer)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# second convolutional layer
	g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	# concatenate merge channel-wise with input layer
	g = Concatenate()([g, input_layer])
	return g

# define the standalone generator model
def define_generator(image_shape, n_resnet=9):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image input
	in_image = Input(shape=image_shape)
	# c7s1-64
	g = Conv2D(64, (7,7), padding='same', kernel_initializer=init)(in_image)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# d128
	g = Conv2D(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# d256
	g = Conv2D(256, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# R256
	for _ in range(n_resnet):
		g = resnet_block(256, g)
	# u128
	g = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# u64
	g = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# c7s1-3
	g = Conv2D(3, (7,7), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	out_image = Activation('tanh')(g)
	# define model
	model = Model(in_image, out_image)
	return model

# define a composite model for updating generators by adversarial and cycle loss
def define_composite_model(g_model_1, d_model, g_model_2, image_shape):
    """Discriminators are compiled and will be trained separately, but generators are not compiled.
    """
	# ensure the model we're updating is trainable
	g_model_1.trainable = True
	# mark discriminator as not trainable
	d_model.trainable = False
	# mark other generator model as not trainable
	g_model_2.trainable = False
	# discriminator element
	input_gen = Input(shape=image_shape)
	gen1_out = g_model_1(input_gen)
	output_d = d_model(gen1_out)
	# identity element
	input_id = Input(shape=image_shape)
	output_id = g_model_1(input_id)
	# forward cycle
	output_f = g_model_2(gen1_out)
	# backward cycle
	gen2_out = g_model_2(input_id)
	output_b = g_model_1(gen2_out)
	# define model graph
	model = Model([input_gen, input_id], [output_d, output_id, output_f, output_b])
	# define optimization algorithm configuration
	opt = Adam(lr=0.0002, beta_1=0.5)
	# compile model with weighting of least squares loss and L1 loss
	model.compile(loss=['mse', 'mae', 'mae', 'mae'], loss_weights=[1, 5, 10, 10], optimizer=opt)
	return model

# load and prepare training images
def load_real_samples(filename):
	# load the dataset
	data = load(filename)
	# unpack arrays
	X1, X2 = data['arr_0'], data['arr_1']
	# scale from [0,255] to [-1,1]
	X1 = (X1 - 127.5) / 127.5
	X2 = (X2 - 127.5) / 127.5
	return [X1, X2]

# select a batch of random samples, returns images and target
def generate_real_samples(dataset, n_samples, patch_shape):
	# choose random instances
	ix = randint(0, dataset.shape[0], n_samples)
	# retrieve selected images
	X = dataset[ix]
	# generate 'real' class labels (1)
	y = ones((n_samples, patch_shape, patch_shape, 1))
	return X, y

# generate a batch of images, returns images and targets
def generate_fake_samples(g_model, dataset, patch_shape):
	# generate fake instance
	X = g_model.predict(dataset)
	# create 'fake' class labels (0)
	y = zeros((len(X), patch_shape, patch_shape, 1))
	return X, y

# save the generator models to file
def save_models(step, g_model_AtoB, g_model_BtoA):
	# save the first generator model
	filename1 = 'g_model_AtoB_%06d.h5' % (step+1)
	g_model_AtoB.save(filename1)
	# save the second generator model
	filename2 = 'g_model_BtoA_%06d.h5' % (step+1)
	g_model_BtoA.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, trainX, name, n_samples=5):
	# select a sample of input images
	X_in, _ = generate_real_samples(trainX, n_samples, 0)
	# generate translated images
	X_out, _ = generate_fake_samples(g_model, X_in, 0)
	# scale all pixels from [-1,1] to [0,1]
	X_in = (X_in + 1) / 2.0
	X_out = (X_out + 1) / 2.0
	# plot real images
	for i in range(n_samples):
		pyplot.subplot(2, n_samples, 1 + i)
		pyplot.axis('off')
		pyplot.imshow(X_in[i])
	# plot translated image
	for i in range(n_samples):
		pyplot.subplot(2, n_samples, 1 + n_samples + i)
		pyplot.axis('off')
		pyplot.imshow(X_out[i])
	# save plot to file
	filename1 = '%s_generated_plot_%06d.png' % (name, (step+1))
	pyplot.savefig(filename1)
	pyplot.close()

# update image pool for fake images
def update_image_pool(pool, images, max_size=50):
	selected = list()
	for image in images:
		if len(pool) < max_size:
			# stock the pool
			pool.append(image)
			selected.append(image)
		elif random() < 0.5:
			# use image, but don't add it to the pool
			selected.append(image)
		else:
			# replace an existing image and use replaced image
			ix = randint(0, len(pool))
			selected.append(pool[ix])
			pool[ix] = image
	return asarray(selected)

# train cyclegan models
def train(d_model_A, d_model_B, g_model_AtoB, g_model_BtoA, c_model_AtoB, c_model_BtoA, dataset):
	# define properties of the training run
	n_epochs, n_batch, = 100, 1
	# determine the output square shape of the discriminator
	n_patch = d_model_A.output_shape[1]
	# unpack dataset
	trainA, trainB = dataset
	# prepare image pool for fakes
	poolA, poolB = list(), list()
	# calculate the number of batches per training epoch
	bat_per_epo = int(len(trainA) / n_batch)
	# calculate the number of training iterations
	n_steps = bat_per_epo * n_epochs
	# manually enumerate epochs
	for i in range(n_steps):
		# select a batch of real samples
		X_realA, y_realA = generate_real_samples(trainA, n_batch, n_patch)
		X_realB, y_realB = generate_real_samples(trainB, n_batch, n_patch)
		# generate a batch of fake samples
		X_fakeA, y_fakeA = generate_fake_samples(g_model_BtoA, X_realB, n_patch)
		X_fakeB, y_fakeB = generate_fake_samples(g_model_AtoB, X_realA, n_patch)
		# update fakes from pool
		X_fakeA = update_image_pool(poolA, X_fakeA)
		X_fakeB = update_image_pool(poolB, X_fakeB)
		# update generator B->A via adversarial and cycle loss
		g_loss2, _, _, _, _  = c_model_BtoA.train_on_batch([X_realB, X_realA], [y_realA, X_realA, X_realB, X_realA])
		# update discriminator for A -> [real/fake]
		dA_loss1 = d_model_A.train_on_batch(X_realA, y_realA)
		dA_loss2 = d_model_A.train_on_batch(X_fakeA, y_fakeA)
		# update generator A->B via adversarial and cycle loss
		g_loss1, _, _, _, _ = c_model_AtoB.train_on_batch([X_realA, X_realB], [y_realB, X_realB, X_realA, X_realB])
		# update discriminator for B -> [real/fake]
		dB_loss1 = d_model_B.train_on_batch(X_realB, y_realB)
		dB_loss2 = d_model_B.train_on_batch(X_fakeB, y_fakeB)
		# summarize performance
		print('>%d, dA[%.3f,%.3f] dB[%.3f,%.3f] g[%.3f,%.3f]' % (i+1, dA_loss1,dA_loss2, dB_loss1,dB_loss2, g_loss1,g_loss2))
		# evaluate the model performance every so often
		if (i+1) % (bat_per_epo * 1) == 0:
			# plot A->B translation
			summarize_performance(i, g_model_AtoB, trainA, 'AtoB')
			# plot B->A translation
			summarize_performance(i, g_model_BtoA, trainB, 'BtoA')
		if (i+1) % (bat_per_epo * 5) == 0:
			# save the models
			save_models(i, g_model_AtoB, g_model_BtoA)

# load image data
dataset = load_real_samples('horse2zebra_256.npz')
print('Loaded', dataset[0].shape, dataset[1].shape)
# define input shape based on the loaded dataset
image_shape = dataset[0].shape[1:]
# generator: A -> B
g_model_AtoB = define_generator(image_shape)
# generator: B -> A
g_model_BtoA = define_generator(image_shape)
# discriminator: A -> [real/fake]
d_model_A = define_discriminator(image_shape)
# discriminator: B -> [real/fake]
d_model_B = define_discriminator(image_shape)
# composite: A -> B -> [real/fake, A]
c_model_AtoB = define_composite_model(g_model_AtoB, d_model_B, g_model_BtoA, image_shape)
# composite: B -> A -> [real/fake, B]
c_model_BtoA = define_composite_model(g_model_BtoA, d_model_A, g_model_AtoB, image_shape)
# train models
train(d_model_A, d_model_B, g_model_AtoB, g_model_BtoA, c_model_AtoB, c_model_BtoA, dataset)

---

➡️ **Next / 下一步**: File 4 of 5